# DCT Laboratory — Volume II, Chapter 5
## Dynamic Enterprise Optimization
**Seed `26205`** · Companion to Chapter 5 and **AXIOM Lab 2.5** (module **AXIOM-05**) · Mirrored in `DCT_V2_Ch05_Lab.xlsx`

This laboratory reproduces **Worked Example 5.1, *When Should Meridian Pay Its Shareholders?***, with the
book's declarations, and works the five steps of AXIOM Lab 2.5: simulate, transcribe, price, move the
horizon and the rate, and stress. Every Meridian number below is the book's (Table 5.3, Figures 5.1–5.3,
Online Appendix 5.B.3); the final cell checks them. Money is in \$ million.

## The declarations (Worked Example 5.1, Step 1)

Volume I's automation program, as in Worked Examples 2.1–4.1. The state is year-end cash $q_k$ and
efficiency $e_k$, from $q_0 = 620$ and $e_0 = 71$. Each year the chief operating officer may run a round,
$z_k \in \{0, 1\}$: it costs \$120 million and closes half the gap to an efficiency of 85. Savings are
$S_k = 10(e_k - 71)$ a year, and the chief financial officer pays out $p_k \ge 0$ at the end of the year, so
$q_k = q_{k-1} - 120z_k + S_k - p_k$. A round needs \$470 million on hand when it starts, every year ends
with at least \$350 million, and year 6 ends with \$470 million, cash for another round. Chapter 5 adds
the board's two declarations: payouts are discounted at its hurdle rate of 10 percent a year,
$\beta = 1/1.1$, and the savings left in place after year 6 continue forever, the lines being maintained.
The value is $\sum_{k=1}^{6} \beta^{k} p_k + \beta^{6} S_6/r$: the payouts, plus the year-6 savings forever.
At 10 percent the perpetuity is $10\,S_6$, the ten years of credit Part I used. Cash earns nothing.

In [ ]:
import itertools
import numpy as np
from scipy.optimize import linprog
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26205

COST, CEIL, VAL = 120.0, 85.0, 10.0        # a round costs $120m and closes half the gap to 85; $10m a year a point
FLOOR, PRE, Q_END = 350.0, 470.0, 470.0    # the floor at every year end; cash a round needs; the end condition
E0, Q0, K, R = 71.0, 620.0, 6, 0.10        # today's efficiency and cash; six years; the board's hurdle rate
SCHEDULES = list(itertools.product([0, 1], repeat=K))     # the 64 schedules of rounds

def beta(r):
    return 1.0 / (1.0 + r)

def efficiency(z, short=0.0, e0=E0):
    # a round closes half the gap to 85 (a point less of it when the rounds fall short)
    e, out = e0, []
    for zk in z:
        if zk:
            e = e + (CEIL - e) / 2 - short
        out.append(e)
    return np.array(out)

def savings(z, short=0.0, e0=E0):
    return VAL * (efficiency(z, short, e0) - E0)

def terminal(s_last, r, kind="perp", years=None):
    # what the plan credits for the savings left in place after its last year
    if kind == "perp":
        return s_last / r                                  # forever, at the hurdle rate
    if kind == "zero":
        return 0.0
    if kind == "life":
        return s_last * (1 - (1 + r) ** -years) / r        # over the life of the lines
    return years * s_last                                  # "credit": Part I's undiscounted years

def war_chest(z, cash_in, q_end=Q_END, start_floor=False):
    # R_k: the least cash at the end of year k from which the rest of the schedule keeps every
    # commitment with no further payout; computed backward from the end condition
    T = len(z)
    Rc = np.empty(T + 1)
    Rc[T] = max(q_end, FLOOR)
    for k in range(T, 0, -1):
        need = Rc[k] + COST * z[k - 1] - cash_in[k - 1]
        if z[k - 1]:
            need = max(need, PRE)                          # the round's precondition
        if k - 1 >= 1 or start_floor:
            need = max(need, FLOOR)                        # the floor at every year end
        Rc[k - 1] = need
    return Rc

def pay_above(z, cash_in, q0=Q0, q_end=Q_END, start_floor=False):
    # pay out, every year end, everything above the war chest; None if today's cash cannot cover it
    Rc = war_chest(z, cash_in, q_end, start_floor)
    if q0 < Rc[0] - 1e-9:
        return None
    q, p, qs = q0, [], [q0]
    for k in range(len(z)):
        q = q - COST * z[k] + cash_in[k]
        pk = max(0.0, q - Rc[k + 1])
        q -= pk
        p.append(pk)
        qs.append(q)
    return np.array(p), np.array(qs), Rc

def plan_value(z, r=R, kind="perp", years=None, short=0.0, extra=None):
    # a schedule of rounds with its best payouts: the payouts discounted at r plus the terminal credit
    S = savings(z, short)
    out = pay_above(z, S if extra is None else S + extra)
    if out is None:
        return None
    p, qs, Rc = out
    w = beta(r) ** np.arange(1, len(z) + 1)
    return float(w @ p + w[-1] * terminal(S[-1], r, kind, years)), p, qs, Rc, S

def best_plan(r=R, kind="perp", years=None, short=0.0, schedules=None):
    # transcription solved exactly: every schedule of rounds, each with its payout program inside
    best = None
    for z in (SCHEDULES if schedules is None else schedules):
        out = plan_value(z, r, kind, years, short)
        if out is not None and (best is None or out[0] > best[0] + 1e-9):
            best = (out[0], tuple(z)) + out[1:]
    return best

gap = CEIL - E0
w6 = beta(R) ** np.arange(1, K + 1)
print(f"beta = {beta(R):.6f} a year; at 10 percent the perpetuity S_6/r is {1 / R:.0f} years of S_6")
print("savings each successive round adds a year:", [VAL * gap / 2 ** n for n in range(1, K + 1)])
print(f"{len(SCHEDULES)} schedules of rounds; payouts discounted at {R:.0%} a year; cash earns nothing")

## Panel 1 — Simulate: the CFO's rule (Proposition 5.1; Table 5.3)

AXIOM Lab 2.5, step 1, and Worked Example 5.1, Step 2. The chief financial officer proposes to pay out,
every year, whatever cash exceeds the \$350 million floor; a planned round runs only if the year opens with
the \$470 million it needs. Rounds are planned for years 1 to 3. The rule is myopic: it decides each year
for that year alone, and so it can break a later commitment even when plans that keep every commitment
exist (Proposition 5.1(i)). Decisions judged strictly year by year do worse still: a round adds at most
\$70 million of savings in its own year against its \$120 million cost, so no round is ever run.

In [ ]:
def cfo_rule(planned=(1, 1, 1, 0, 0, 0), short=0.0, r=R):
    # pay out everything above the floor; run a planned round only if the year opens with $470m
    q, e, zs, p, qs, S = Q0, E0, [], [], [Q0], []
    for k in range(K):
        run = bool(planned[k]) and q >= PRE - 1e-9
        if run:
            q -= COST
            e = e + (CEIL - e) / 2 - short
        s = VAL * (e - E0)
        q += s
        pk = max(0.0, q - FLOOR)
        q -= pk
        zs.append(int(run)); p.append(pk); qs.append(q); S.append(s)
    p = np.array(p)
    w = beta(r) ** np.arange(1, K + 1)
    return dict(z=tuple(zs), p=p, q=np.array(qs), S=np.array(S), end_ok=q >= Q_END - 1e-9,
                value=float(w @ p + w[-1] * S[-1] / r))

cfo = cfo_rule()
print("year  round  savings  payout  year-end cash")
for k in range(K):
    print(f"{k + 1:4d}  {cfo['z'][k]:5d}  {cfo['S'][k]:7.1f}  {cfo['p'][k]:6.1f}  {cfo['q'][k + 1]:13.1f}")
print(f"rounds run: {sum(cfo['z'])}: the year-1 payout leaves {cfo['q'][1]:.0f}, and a round needs {PRE:.0f} on hand")
print(f"year 6 ends at {cfo['q'][-1]:.0f}, {Q_END - cfo['q'][-1]:.0f} short of the end condition; "
      f"worth {cfo['value']:.2f} if the board waived it")
myopic = cfo_rule(planned=(0,) * K)
print(f"judged year by year: rounds {myopic['z']}, payouts {myopic['p']}, worth {myopic['value']:.2f}, end condition missed")
print(f"the most a round adds in its own year: {VAL * gap / 2:.0f} of savings against its cost of {COST:.0f}")

## Panel 2 — Transcribe and solve: 64 schedules of rounds (Theorem 5.1; Proposition 5.3(ii); Table 5.3)

AXIOM Lab 2.5, step 2, and Worked Example 5.1, Step 3. Transcribed, the six years are one program with six
lumpy rounds and a linear payout program inside. For a schedule of rounds the best payouts pay out, every
year, whatever exceeds the **war chest** $R_k$, the least year-end cash from which the rest of the schedule
keeps every commitment with no further payout: $R_6 = 470$ and $R_{k-1} = \max\{R_k + 120z_k - S_k,\ 470z_k,\ 350\}$,
with no floor term at $k = 1$. Payouts have linear value and cash earns nothing, so at any positive hurdle
rate they come as early as the commitments allow (Proposition 5.3(ii)). The cell first checks the war chest
against the payout program solved as a linear program by SciPy's HiGHS solver, for every schedule at three
hurdle rates and in both scenarios, then compares the 64 schedules.

In [ ]:
def lp_payouts(z, r=R, short=0.0):
    # max sum beta^k p_k  s.t.  q_k = q_(k-1) - 120 z_k + S_k - p_k,  p_k >= 0,  q_k >= 350,
    # q_(k-1) >= 470 before a round, q_6 >= 470.  Variables (p_1..p_6, q_1..q_6).
    S, b = savings(z, short), beta(r)
    if z[0] and Q0 < PRE:
        return None
    c = np.concatenate([-b ** np.arange(1, K + 1), np.zeros(K)])
    A = np.zeros((K, 2 * K))
    rhs = S - COST * np.array(z, float)
    rhs[0] += Q0
    for k in range(K):
        A[k, k] = A[k, K + k] = 1.0
        if k:
            A[k, K + k - 1] = -1.0
    lb = [max(FLOOR, PRE if k + 1 < K and z[k + 1] else 0.0, Q_END if k == K - 1 else 0.0) for k in range(K)]
    res = linprog(c, A_eq=A, b_eq=rhs, bounds=[(0, None)] * K + [(l, None) for l in lb], method="highs")
    if res.status != 0:
        return None
    return dict(value=-res.fun + b ** K * S[-1] / r, p=res.x[:K], q=res.x[K:], lb=np.array(lb),
                lam=-res.eqlin.marginals, bound_price=res.lower.marginals[K:])

lp_cases, lp_worst = 0, 0.0
for r in (0.05, 0.10, 0.20):
    for sh in (0.0, 1.0):
        for z in SCHEDULES:
            wc, lp = plan_value(z, r, short=sh), lp_payouts(z, r, sh)
            assert (wc is None) == (lp is None)
            if wc is not None:
                lp_cases += 1
                lp_worst = max(lp_worst, abs(wc[0] - lp["value"]))
print(f"war chest against the linear program: {lp_cases} cases, largest difference {lp_worst:.1e}")

ranked = sorted(((plan_value(z)[0], z) for z in SCHEDULES if plan_value(z) is not None), reverse=True)
print(f"{len(ranked)} of 64 schedules keep every commitment; the best three:")
for v, z in ranked[:3]:
    print(f"   rounds {z}  {v:9.2f}")
best_val, z_best, p_best, q_best, R_best, S_best = best_plan()
e_best = efficiency(z_best)
print("year  round  efficiency  savings  war chest  payout  year-end cash")
for k in range(K):
    print(f"{k + 1:4d}  {z_best[k]:5d}  {e_best[k]:10.2f}  {S_best[k]:7.1f}  {R_best[k + 1]:9.1f}  {p_best[k]:6.1f}  {q_best[k + 1]:13.1f}")
pv_pay, pv_term = float(w6 @ p_best), w6[-1] * S_best[-1] / R
print(f"value {best_val:.2f} = {pv_pay:.2f} of payouts + {pv_term:.2f} of savings after year 6 "
      f"({S_best[-1]:.1f} a year forever, {S_best[-1] / R:,.0f} at year 6)")
at_end = w6[-1] * p_best.sum() + pv_term
print(f"the same {p_best.sum():.0f} paid at the end of year 6 is worth {at_end:.2f}: front-loading adds {best_val - at_end:.2f}")
print(f"the CFO's rule, even with the end condition waived, is worth {best_val - cfo['value']:.2f} less")

yrs = np.arange(K + 1)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axhline(FLOOR, color="#999999", lw=0.8, ls="--")
ax.axhline(PRE, color="#999999", lw=0.8, ls="--")
ax.plot(yrs, q_best, color="#1F4E79", lw=2, marker="o", ms=4, zorder=4, label="best plan: pay out above the war chest")
ax.plot(yrs, cfo["q"], color="#E65100", lw=2, ls="--", marker="s", ms=4, zorder=3, label="CFO's rule: pay out above the floor")
ax.plot([K], [cfo["q"][-1]], marker="x", ms=10, mew=2, color="#E65100", ls="none")
ax.text(1.1, 336, "no second round", fontsize=8, color="#E65100", va="top")
ax.text(6.0, 336, "end missed", fontsize=8, color="#E65100", va="top", ha="right")
ax.set(xlabel="end of year", ylabel="year-end cash ($ million)", ylim=(300, 650), yticks=[350, 470, 620],
       title="The CFO's rule stops the program after one round (seed 26205)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Panel 3 — Price: the multipliers of the dynamics (Theorem 5.1(iii); Section 5.4; Figure 5.1)

AXIOM Lab 2.5, step 3, and Worked Example 5.1, Step 4. In the transcribed program the multipliers of the
dynamics price each stock at each date: the value today of one more dollar of cash, and of one more point of
efficiency, at the end of each year. The prices of cash come from the linear program; those of efficiency
follow the adjoint recursion $\mu_k = 10\lambda_k + (1 - z_{k+1}/2)\,\mu_{k+1}$, with $\mu_6 = 10\lambda_6 + 10\beta^{6}/r$.
The lab asks you to check two of them by perturbing the instance; the cell checks all twelve, then perturbs
every year's cash at once, a thousand times at random. Two prices carry the board's decisions: the round
test, and the price of the end condition.

In [ ]:
lp = lp_payouts(z_best)
lam, bprice = lp["lam"], lp["bound_price"]
mu = np.zeros(K)
mu[-1] = VAL * lam[-1] + beta(R) ** K * VAL / R
for k in range(K - 2, -1, -1):
    mu[k] = VAL * lam[k] + (1 - z_best[k + 1] / 2) * mu[k + 1]

def value_with(z=z_best, extra=None, bump=None):
    # the rounds held fixed, the payouts re-optimized: extra cash by year, or a bump (year, h) to efficiency
    e = efficiency(z)
    if bump is not None:
        j, h = bump
        x = e[j] + h
        e = e.copy(); e[j] = x
        for k in range(j + 1, K):
            if z[k]:
                x = x + (CEIL - x) / 2
            e[k] = x
    S = VAL * (e - E0)
    p, _, _ = pay_above(z, S if extra is None else S + extra)
    return float(w6 @ p + w6[-1] * S[-1] / R)

lam_fd = np.array([value_with(extra=np.eye(K)[k]) - best_val for k in range(K)])
mu_fd = np.array([(value_with(bump=(k, 1e-4)) - best_val) / 1e-4 for k in range(K)])
print("year   cash: price  beta^k  perturbed    efficiency: price  perturbed  in year-k dollars")
for k in range(K):
    print(f"{k + 1:4d}  {lam[k]:11.4f}  {w6[k]:6.4f}  {lam_fd[k]:9.4f}  {mu[k]:18.2f}  {mu_fd[k]:9.2f}  {mu[k] / w6[k]:17.2f}")
print("a dollar arriving in year 2 is worth as much as one in year 1: it shrinks the year-1 war chest")
names = ["q_1 >= 470 (round 2)", "q_2 >= 470 (round 3)", "q_3 >= 350 (floor)", "q_4 >= 350 (floor)",
         "q_5 >= 350 (floor)", "q_6 >= 470 (end condition)"]
for k in range(K):
    if abs(q_best[k + 1] - lp["lb"][k]) < 1e-9:
        print(f"binding: {names[k]:27s} price {bprice[k]:.4f} a dollar")
end_cost = (Q_END - FLOOR) * bprice[-1]
print(f"the end condition holds {Q_END - FLOOR:.0f} above the floor at {bprice[-1]:.4f} a dollar: {end_cost:.2f} today")

rng = np.random.default_rng(SEED)
deltas = rng.uniform(-1.0, 1.0, size=(1000, K))
err = max(abs(value_with(extra=d) - best_val - lam @ d) for d in deltas)
print(f"1,000 random changes of up to $1 million in each year's cash (seed {SEED}): "
      f"the value moves by the priced sum, largest error {err:.1e}")

delta = [VAL * gap / 2 ** n for n in range(1, K + 1)]
for n in (3, 4):
    print(f"round {n}: adds {delta[n - 1]} a year, worth {delta[n - 1] / R:.1f} at 10 percent against {COST:.0f}: "
          + ("pays" if delta[n - 1] / R >= COST else "does not pay"))

## Panel 4 — Move the horizon and the rate (Theorem 5.2; Figure 5.2)

AXIOM Lab 2.5, step 4, and Worked Example 5.1, Step 5. First the hurdle rate, from 3 to 30 percent, with the
savings valued forever. A lower rate buys more rounds, and the rate below which another round pays is found
by bisection. Each round closes half the remaining gap, so each switch point is about half the last; from
the fourth round on it is exactly the round's yield, the savings it adds each year over its \$120 million cost.

In [ ]:
by_rate = [(r, best_plan(r)) for r in (0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.30)]
print("hurdle rate  rounds  schedule              value")
for r, b in by_rate:
    print(f"{100 * r:10.0f}%  {sum(b[1]):6d}  {str(b[1]):20s}  {b[0]:9.2f}")
b5 = dict(by_rate)[0.05]
print("at 5 percent: payouts", b5[2], "(the perpetuity is twenty years of S_6, Chapter 4's credit)")

def n_rounds(r):
    return sum(best_plan(r)[1])

grid = np.linspace(0.005, 0.9, 400)
ns = [n_rounds(r) for r in grid]
switches = []
for a, c, na, nc in zip(grid[:-1], grid[1:], ns[:-1], ns[1:]):
    if na != nc:
        for _ in range(60):
            m = 0.5 * (a + c)
            if n_rounds(m) == na:
                a = m
            else:
                c = m
        switches.append((na, nc, 0.5 * (a + c)))
yields = [d / COST for d in delta]
for na, nc, r in switches:
    print(f"round {na} pays only below {100 * r:7.3f} percent; its yield {delta[na - 1]}/120 is {100 * yields[na - 1]:7.3f} percent")

xs = [1.0] + [100 * s[2] for s in switches] + [60.0]
ys = [n_rounds(0.01)] + [s[1] for s in switches]
ys = ys + [ys[-1]]
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.step(xs, ys, where="post", color="#1F4E79", lw=2)
ax.set_xscale("log")
ax.set_xticks([1, 2, 5, 10, 20, 50]); ax.set_xticklabels(["1", "2", "5", "10", "20", "50"]); ax.minorticks_off()
for na, nc, r in switches:
    ax.text(100 * r * 1.06, na - 0.5, f"{100 * r:.1f}", fontsize=8, color="#1F4E79", va="center")
for x, txt in ((5.0, "20 years'\ncredit"), (10.0, "10 years'\ncredit")):
    n = n_rounds(x / 100)
    ax.plot([x, x], [n - 1.05, n - 0.12], color="#999999", lw=0.8, zorder=1)
    ax.plot([x], [n], marker="o", ms=6, mfc="white", mec="#616161", mew=1.2, zorder=5)
    ax.text(x, n - 1.5, txt, fontsize=8, color="#616161", ha="center", va="center")
ax.set(xlim=(1, 60), ylim=(-0.3, 6.8), xlabel="hurdle rate (percent a year, log scale)",
       ylabel="rounds in the best plan", title="Each further round pays only below about half the rate (seed 26205)")
plt.tight_layout(); plt.show()

Then the terminal value and the window. With no terminal value a plan counts only the payouts inside its
window, so an investment whose savings fall after the horizon is worth nothing to it (Theorem 5.2(iii));
with the savings valued forever, the value of continuing, truncation is exact (Theorem 5.2(i)). Each window
keeps the end condition at its end. Full enumeration confirms that the rounds come in the first years for
windows of up to twelve years; longer windows compare those schedules.

In [ ]:
zero = best_plan(kind="zero")
zero_true = plan_value(zero[1])[0]
print(f"six years, no terminal value: rounds {zero[1]}, payouts {zero[2]}")
print(f"   worth {zero[0]:.2f} by its own criterion and {zero_true:.2f} with the savings forever: {best_val - zero_true:.2f} less")
life = best_plan(kind="life", years=20)
credit = best_plan(r=1e-12, kind="credit", years=10)
print(f"savings over the twenty-year life of the lines: rounds {life[1]}, worth {life[0]:.2f}")
print(f"undiscounted with ten years of credit (Part I): rounds {credit[1]}, {credit[0]:.2f}")

def window_plan(T, kind, early=False):
    scheds = [tuple([1] * n + [0] * (T - n)) for n in range(T + 1)] if early else itertools.product([0, 1], repeat=T)
    return best_plan(kind=kind, schedules=scheds)

windows = []
for T in range(1, 21):
    out = []
    for kind in ("zero", "perp"):
        b = window_plan(T, kind, early=True)
        if T <= 12:
            full = window_plan(T, kind)
            assert abs(full[0] - b[0]) < 1e-6 and sum(full[1]) == sum(b[1])
        out.append(sum(b[1]))
    windows.append((T, out[0], out[1]))
print("window (years):        " + "".join(f"{T:3d}" for T, _, _ in windows))
print("rounds, no terminal:   " + "".join(f"{nz:3d}" for _, nz, _ in windows))
print("rounds, forever:       " + "".join(f"{npp:3d}" for _, _, npp in windows))
third_zero = min(T for T, nz, _ in windows if nz >= 3)
print(f"no terminal value: the third round appears only with a window of {third_zero} years")

Ts = [t for t, _, _ in windows]
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.step(Ts, [nz for _, nz, _ in windows], where="mid", color="#E65100", lw=2, label="no terminal value")
ax.step(Ts, [npp for _, _, npp in windows], where="mid", color="#1F4E79", lw=2, ls="--", label="savings valued forever")
ax.annotate("third round only\nfrom 15 years", xy=(15, 3), xytext=(9.5, 3.7), fontsize=8, color="#E65100",
            ha="center", va="center", arrowprops=dict(arrowstyle="-", color="#E65100", lw=0.8))
ax.set(xlim=(0.5, 20.5), ylim=(-0.3, 4.3), xticks=[1, 5, 10, 15, 20], yticks=range(5),
       xlabel="planning window (years)", ylabel="rounds in the best plan",
       title="No terminal value: the third round needs a 15-year window (seed 26205)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 5 — Re-plan: a present-biased board and rolling windows (Theorem 5.3; Table 5.3)

With exponential discounting, a company that re-plans every year from where it stands and carries out the
first decision follows a best plan (Theorem 5.3(iii)). Section 5.6's board instead weighs this year's payout
in full and every later year's at a share $b$ of its discounted value, the pull of this year's results. A
round's war chest is withheld from this year's payout while its savings come later, so a round always looks
better a year on (Theorem 5.3(iv)). Shareholders discount at 10 percent throughout. Re-planning over
rolling six-year windows has the same weakness without a terminal value, and none with the perpetuity.

In [ ]:
def best_from(q, e, years, weights, kind="perp", short=0.0, start_floor=False):
    # the best schedule for the remaining years from cash q and efficiency e, under the given payout weights
    best = None
    for z in itertools.product([0, 1], repeat=years):
        S = savings(z, short, e0=e)
        out = pay_above(z, S, q0=q, start_floor=start_floor)
        if out is None:
            continue
        val = float(np.asarray(weights) @ out[0] + weights[-1] * terminal(S[-1], R, kind))
        if best is None or val > best[0] + 1e-9:
            best = (val, z, out[0], out[2])
    return best

def present_bias(b, short=0.0):
    # at the start of each year: weigh that year 1 and the year i later b beta^i, re-plan, carry out the first year
    q, e, zs, pays, plans = Q0, E0, [], [], []
    for j in range(K):
        L = K - j
        _, z, p, _ = best_from(q, e, L, [1.0] + [b * beta(R) ** i for i in range(1, L)], short=short)
        plans.append(z)
        if z[0]:
            q -= COST
            e = e + (CEIL - e) / 2 - short
        q += VAL * (e - E0) - p[0]
        zs.append(z[0]); pays.append(p[0])
    pays = np.array(pays)
    return dict(z=tuple(zs), p=pays, plans=plans, q_end=q, value=float(w6 @ pays + w6[-1] * VAL * (e - E0) / R))

exp_board = present_bias(1.0)
print(f"b = 1 (exponential): carries out {exp_board['z']}, worth {exp_board['value']:.2f}, the best plan")
pb9, pb6 = present_bias(0.9), present_bias(0.6)
for b, res in ((0.9, pb9), (0.6, pb6)):
    print(f"b = {b}: plans by date {res['plans']}")
    print(f"       carries out rounds {res['z']}, payouts {res['p']}, ends with {res['q_end']:.0f}, "
          f"worth {res['value']:.2f} ({best_val - res['value']:.2f} less)")

def carried(b):
    res = present_bias(b)
    return res["z"], tuple(np.round(res["p"], 6))

grid_b = np.linspace(0.50, 0.999, 500)
keys = [carried(b) for b in grid_b]
bias_bounds = []
for a, c, ka, kc in zip(grid_b[:-1], grid_b[1:], keys[:-1], keys[1:]):
    if ka != kc:
        while c - a > 1e-10:
            m = 0.5 * (a + c)
            if carried(m) == ka:
                a = m
            else:
                c = m
        bias_bounds.append((0.5 * (a + c), ka[0], kc[0]))
edges = [grid_b[0]] + [bb for bb, _, _ in bias_bounds] + [1.0]
print("weight b on later years   rounds carried out   payouts, years 1-6                value     loss")
for lo, hi in zip(edges[:-1], edges[1:]):
    res = present_bias(0.5 * (lo + hi))
    yrs_run = ", ".join(str(k + 1) for k in range(K) if res["z"][k])
    pays_txt = ", ".join(f"{x:g}" for x in np.round(res["p"], 2))
    print(f"{lo:.5f} to {hi:.5f}      {yrs_run:19s}  {pays_txt:32s}  {res['value']:8.2f}  {best_val - res['value']:7.2f}")
b_keep = bias_bounds[-1][0]
print(f"only a board above b = {b_keep:.5f} carries out the plan it approved")

def rolling(T, kind, years=30):
    # re-plan every year over a T-year window (end condition at its end); carry out the first year.
    # Value: payouts for thirty years, then the savings forever and the cash above 470 paid out.
    q, e, zs, pays, s = Q0, E0, [], [], 0.0
    for j in range(years):
        _, z, p, _ = best_from(q, e, T, [beta(R) ** i for i in range(T)], kind)
        if z[0]:
            q -= COST
            e = e + (CEIL - e) / 2
        s = VAL * (e - E0)
        q += s - p[0]
        zs.append(z[0]); pays.append(p[0])
    val = sum(beta(R) ** (k + 1) * pays[k] for k in range(years)) + beta(R) ** years * (s / R + q - Q_END)
    return sum(zs), tuple(zs[:6]), val

roll_zero, roll_perp = rolling(6, "zero"), rolling(6, "perp")
print(f"rolling six-year windows, no terminal value: {roll_zero[0]} rounds {roll_zero[1]}, worth {roll_zero[2]:.2f}")
print(f"rolling six-year windows, savings forever:   {roll_perp[0]} rounds {roll_perp[1]}, worth {roll_perp[2]:.2f}")

## Panel 6 — Stress: each round a point short (Theorem 5.3; Figure 5.3; Table 5.3)

AXIOM Lab 2.5, step 5, and Worked Example 5.1, Step 6. The program office warns that each round may close a
point less of the gap than planned (Volume I, Worked Example 6.1). Paying Step 3's schedule regardless runs a
planned round only if the year opens with \$470 million. The **war-chest rule** decides year 1's round on the
forecast; from the end of year 1 it recomputes, each year end, the cash the rest of the program needs from the
measured efficiency and the revised estimate, and pays out the rest. The theorem is deterministic: when the
future is uncertain, a rule can use information that a plan cannot.

In [ ]:
def schedule_regardless(z, p, short=1.0):
    q, e, qs, ran, breaches = Q0, E0, [Q0], [], []
    for k in range(K):
        run = bool(z[k]) and q >= PRE - 1e-9
        if z[k] and not run:
            breaches.append(f"round {k + 1} cannot start ({q:.0f} < {PRE:.0f})")
        if run:
            q -= COST
            e = e + (CEIL - e) / 2 - short
        q += VAL * (e - E0) - p[k]
        if q < FLOOR - 1e-9:
            breaches.append(f"floor breached in year {k + 1} ({q:.1f})")
        qs.append(q); ran.append(int(run))
    if q < Q_END - 1e-9:
        breaches.append(f"end condition missed ({q:.1f})")
    return np.array(qs), tuple(ran), breaches

def war_chest_rule(short=1.0):
    # year 1's round on the forecast; at each year end, re-plan the rest of the program from the cash on hand,
    # the measured efficiency and the revised estimate, and pay out everything above its war chest
    q, e = Q0, E0
    zs = [best_plan()[1][0]]
    if zs[0]:
        q -= COST
        e = e + (CEIL - e) / 2 - short
    q += VAL * (e - E0)
    pays, qs = [], [Q0]
    for j in range(1, K + 1):
        L = K - j
        if L == 0:
            p = max(0.0, q - Q_END)
        else:
            _, z, _, Rc = best_from(q, e, L, [beta(R) ** i for i in range(1, L + 1)], short=short, start_floor=True)
            p = max(0.0, q - Rc[0])
        q -= p
        pays.append(p); qs.append(q)
        if L == 0:
            break
        if z[0]:
            q -= COST
            e = e + (CEIL - e) / 2 - short
        zs.append(z[0])
        q += VAL * (e - E0)
    pays = np.array(pays)
    return dict(z=tuple(zs), p=pays, q=np.array(qs), value=float(w6 @ pays + w6[-1] * VAL * (e - E0) / R))

q_fixed, ran_fixed, breaches = schedule_regardless(z_best, p_best)
print("Step 3's schedule paid regardless: year-end cash", q_fixed[1:])
print(f"   rounds run {ran_fixed}; breaches: " + "; ".join(breaches))
rule = war_chest_rule()
print("war-chest rule: rounds", rule["z"], "payouts", rule["p"], "year-end cash", rule["q"][1:])
known = best_plan(short=1.0)
print(f"   worth {rule['value']:.2f}; the best plan with the shortfall known from the start: rounds {known[1]}, "
      f"payouts {known[2]}, {known[0]:.2f}")
cfo_short = cfo_rule(short=1.0)
print(f"the CFO's rule under the shortfall: payouts {cfo_short['p']}, worth {cfo_short['value']:.2f}, end condition missed")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axhline(FLOOR, color="#999999", lw=0.8, ls="--")
ax.axhline(PRE, color="#999999", lw=0.8, ls="--")
ax.plot(yrs, rule["q"], color="#1F4E79", lw=2, marker="o", ms=4, label="war-chest rule")
ax.plot(yrs, q_fixed, color="#B71C1C", lw=2, ls="--", marker="s", ms=4, label="Step 3's schedule")
ax.plot([5, 6], q_fixed[5:], marker="x", ms=10, mew=2, color="#B71C1C", ls="none")
ax.annotate("third round\nblocked", xy=(2, q_fixed[2]), xytext=(1.0, 395), fontsize=8, color="#B71C1C",
            ha="center", va="center", arrowprops=dict(arrowstyle="-", color="#B71C1C", lw=0.8))
ax.text(6.0, 318, "floor breached, end missed", fontsize=8, color="#B71C1C", ha="right", va="center")
ax.set(xlabel="end of year", ylabel="year-end cash ($ million)", ylim=(300, 650), yticks=[350, 470, 620],
       title="A point short: the rule keeps every commitment (seed 26205)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

def years_of(z):
    return ", ".join(str(k + 1) for k in range(len(z)) if z[k])

def payouts(p):
    return ", ".join(f"{x:g}" for x in np.round(p, 2))

table53 = [("CFO's rule", years_of(cfo["z"]), payouts(cfo["p"]), cfo["value"], "end missed"),
           ("Best plan", years_of(z_best), payouts(p_best), best_val, "kept"),
           ("No terminal value", years_of(zero[1]), payouts(zero[2]), zero_true, "kept"),
           ("Present-biased board", years_of(pb9["z"]), payouts(pb9["p"]), pb9["value"], "kept"),
           ("Step 3's schedule, a point short", years_of(ran_fixed), payouts(p_best), None, "floor, end breached"),
           ("War-chest rule, a point short", years_of(rule["z"]), payouts(rule["p"]), rule["value"], "kept")]
print("\nTable 5.3: payout plans compared (value at 10 percent: payouts plus the year-6 savings forever)")
print(f"{'plan':34s} {'rounds':9s} {'payouts, years 1-6':32s} {'value':>9s}  commitments")
for name, yrs_, pays_, v, ok in table53:
    print(f"{name:34s} {yrs_:9s} {pays_:32s} {('%9.2f' % v) if v is not None else '       --'}  {ok}")
print("(the CFO's rule is valued as if the board waived its end condition)")

## Validation — the book's numbers, and agreement with `DCT_V2_Ch05_Lab.xlsx`

Each checkpoint is a number printed in Chapter 5 (Worked Example 5.1, Table 5.3, Figures 5.1–5.3) or in
Online Appendix 5.B.3. The workbook recomputes the ones marked *workbook* with spreadsheet formulas and
shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
checks = [
    ("best plan: value (workbook)", best_val, 1021.93, 5e-3),
    ("best plan: payouts, present value (workbook)", pv_pay, 330.45, 5e-3),
    ("best plan: savings after year 6, today (workbook)", pv_term, 691.48, 5e-3),
    ("best plan: total paid out (workbook)", p_best.sum(), 455.0, 1e-9),
    ("war chest at the end of year 1 (workbook)", R_best[1], 485.0, 1e-9),
    ("front-loading against paying at year 6 (workbook)", best_val - at_end, 73.62, 5e-3),
    ("CFO's rule, end condition waived (workbook)", cfo["value"], 836.36, 5e-3),
    ("CFO's rule falls short of the best plan (workbook)", best_val - cfo["value"], 185.6, 0.05),
    ("plan judged year by year (workbook)", myopic["value"], 245.45, 5e-3),
    ("war chest = linear program: cases", lp_cases, 384, 0),
    ("war chest = linear program: largest difference", lp_worst, 0.0, 1e-12),
    ("price of cash, year 2 (workbook)", lam[1], 0.909, 5e-4),
    ("price of cash, year 6 (workbook)", lam[5], 0.564, 5e-4),
    ("price of round 3's precondition (workbook)", bprice[1], 0.158, 5e-4),
    ("cost of the end condition (workbook)", end_cost, 67.74, 5e-3),
    ("price of a point of efficiency, year 1 (workbook)", mu[0], 34.30, 5e-3),
    ("price of a point of efficiency, year 3 (workbook)", mu[2], 82.64, 5e-3),
    ("round 3's savings at 10 percent (workbook)", delta[2] / R, 175.0, 1e-9),
    ("no terminal value: own value (workbook)", zero[0], 384.17, 5e-3),
    ("no terminal value: with the savings forever (workbook)", zero_true, 976.87, 5e-3),
    ("savings over the twenty-year life (workbook)", life[0], 919.15, 5e-3),
    ("value at 5 percent (workbook)", b5[0], 2261.14, 5e-3),
    ("third round pays below (percent) (workbook)", 100 * switches[3][2], 14.55, 5e-3),
    ("fourth round pays below (percent) (workbook)", 100 * switches[2][2], 7.292, 5e-4),
    ("window for a third round, no terminal value (workbook)", third_zero, 15, 0),
    ("present-biased board (b = 0.9): value (workbook)", pb9["value"], 1007.55, 5e-3),
    ("present-biased board (b = 0.9): loss (workbook)", best_val - pb9["value"], 14.39, 5e-3),
    ("board at b = 0.6: loss", best_val - pb6["value"], 102.43, 5e-3),
    ("least b that carries out the approved plan", b_keep, 0.94286, 5e-6),
    ("rolling six-year windows, no terminal value", roll_zero[2], 1038.58, 5e-3),
    ("rolling six-year windows, savings forever", roll_perp[2], 1082.79, 5e-3),
    ("a point short: Step 3's schedule, cash end of year 2 (workbook)", q_fixed[2], 445.0, 1e-9),
    ("a point short: Step 3's schedule, cash end of year 5 (workbook)", q_fixed[5], 347.5, 1e-9),
    ("a point short: war-chest rule (workbook)", rule["value"], 853.73, 5e-3),
    ("a point short: best plan with the shortfall known (workbook)", known[0], 853.73, 5e-3),
    ("a point short: CFO's rule (workbook)", cfo_short["value"], 736.36, 5e-3),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"
    ok += status == "PASS"
    print(f"{name:64s} {got:11.4f}   book {want:10.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 6 lets the review dates come continuously: this chapter's prices become the costates of
optimal control, and one rule replaces the whole program, Pontryagin's maximum principle: at every moment,
choose the decision that maximizes the Hamiltonian, the value earned now plus the value banked in the stocks
at their prices. AXIOM-05 runs the same instance interactively (Lab 2.5); this notebook and the workbook are
its reproducible record.